In [16]:
import pandas as pd 
import sys 
import numpy as np 

In [17]:
file_path = "../data/raw/張瓊之_給馬老師資料20250925_V4(遺失值不用登錄成-1).xlsx"

demo = pd.read_excel(file_path, sheet_name="demographic")
onset = pd.read_excel(file_path, sheet_name="發病年紀")
mmse = pd.read_excel(file_path, sheet_name="MMSE longitudinal")
blood = pd.read_excel(file_path, sheet_name="blood_data_20250921")
casi = pd.read_excel(file_path, sheet_name="CASI longitudinal")


### 處理欄位: 命名欄位

In [18]:
# =============================================================================
# 1. 清理 demographic
# =============================================================================

# 第一列是英文欄位名稱，不是患者資料
demo = demo.iloc[1:].copy()

demo.columns = [
    "patient_id",
    "diagnosis",
    "bio_category",
    "apoe",
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia"
]

# =============================================================================
# 2. 清理 onset
# =============================================================================

onset = onset.rename(columns={
    "Count Number": "patient_id",
    "age of onset": "age_onset",
    "Biological category": "bio_category_onset"
})

# =============================================================================
# 3. 清理 MMSE
# =============================================================================

mmse = mmse.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MMSE(numerical)": "mmse",
    "CDR (scale)": "cdr_global",
    "CDR_M (scale)": "cdr_memory",
    "CDR_O (scale)": "cdr_orientation",
    "CDR_J (scale)": "cdr_judgment",
    "CDR_C (scale)": "cdr_community",
    "CDR_H (scale)": "cdr_home_hobbies",
    "CDR_P (scale)": "cdr_personal_care",
    "CDR-SOB(numerical)": "cdr_sob"
})

# =============================================================================
# 4. 清理 CASI
# =============================================================================

casi = casi.rename(columns={
    "Count Number": "patient_id",
    "Date": "casi_date",
    "MENMA10": "casi_mental_manipulation",
    "ATTEN8": "casi_attention",
    "ORIEN18": "casi_orientation",
    "LTM10": "casi_long_term_memory",
    "STM12": "casi_short_term_memory",
    "ABSTR12": "casi_abstraction",
    "DRAW10": "casi_drawing",
    "ANML10": "casi_verbal_fluency",
    "LANG10": "casi_language",
    "Total": "casi_total"
})

# =============================================================================
# 5. 清理 blood
# =============================================================================

blood = blood.rename(columns={
    "Count": "patient_id",
    "Date": "blood_date",
    "HDL-C": "hdl",
    "VLDL-C": "vldl",
    "LDL-C": "ldl",
    "T-Cholesterol": "total_cholesterol",
    "Triglyceride": "triglyceride",
    "AC sugar level": "fasting_glucose",
    "HbA1c": "hba1c"
})

### 空值檢查

In [19]:
tables = {
    "demographic": demo,
    "發病年紀": onset,
    "MMSE longitudinal": mmse,
    "blood_data_20250921": blood,
    "CASI longitudinal": casi,
}

for sheet_name, df in tables.items():
    missing_ratio = df.isna().mean().sort_values(ascending=False) * 100
    result = pd.DataFrame({
        "column": missing_ratio.index,
        "missing_ratio_%": missing_ratio.values
    })
    
    print(f"\n=== {sheet_name} ===")
    print(result.to_string(index=False))


=== demographic ===
        column  missing_ratio_%
          apoe        14.911606
hyperlipidemia        10.914681
      diabetes        10.914681
  hypertension        10.914681
     diagnosis         0.000000
    patient_id         0.000000
age_first_mmse         0.000000
        gender         0.000000
  bio_category         0.000000
     education         0.000000

=== 發病年紀 ===
            column  missing_ratio_%
         age_onset        11.837048
        patient_id         0.000000
bio_category_onset         0.000000

=== MMSE longitudinal ===
           column  missing_ratio_%
     cdr_judgment         0.900901
  cdr_orientation         0.900901
       cdr_memory         0.900901
cdr_personal_care         0.900901
 cdr_home_hobbies         0.900901
          cdr_sob         0.900901
    cdr_community         0.900901
             mmse         0.000000
       visit_date         0.000000
       patient_id         0.000000
       cdr_global         0.000000

=== blood_data_202509

### 統一ID、數值與日期型別

In [20]:
# =============================================================================
# 6. 統一 patient_id
# =============================================================================

dataframes = [demo, onset, mmse, casi, blood]

for df in dataframes:
    df["patient_id"] = pd.to_numeric(
        df["patient_id"],
        errors="coerce"
    ).astype("Int64")

# 移除無法辨識 ID 的資料
for df in dataframes:
    df.dropna(subset=["patient_id"], inplace=True)

# =============================================================================
# 7. 日期轉換
# =============================================================================

mmse["visit_date"] = pd.to_datetime(
    mmse["visit_date"],
    errors="coerce"
)

casi["casi_date"] = pd.to_datetime(
    casi["casi_date"],
    errors="coerce"
)

blood["blood_date"] = pd.to_datetime(
    blood["blood_date"],
    errors="coerce"
)

# =============================================================================
# 8. 數值欄位轉換
# =============================================================================

demo_numeric_cols = [
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia"
]

onset_numeric_cols = ["age_onset"]

mmse_numeric_cols = [
    "mmse",
    "cdr_global",
    "cdr_memory",
    "cdr_orientation",
    "cdr_judgment",
    "cdr_community",
    "cdr_home_hobbies",
    "cdr_personal_care",
    "cdr_sob"
]

casi_numeric_cols = [
    "casi_mental_manipulation",
    "casi_attention",
    "casi_orientation",
    "casi_long_term_memory",
    "casi_short_term_memory",
    "casi_abstraction",
    "casi_drawing",
    "casi_verbal_fluency",
    "casi_language",
    "casi_total"
]

blood_numeric_cols = [
    "hdl",
    "vldl",
    "ldl",
    "total_cholesterol",
    "triglyceride",
    "fasting_glucose",
    "hba1c"
]

for col in demo_numeric_cols:
    demo[col] = pd.to_numeric(demo[col], errors="coerce")

for col in onset_numeric_cols:
    onset[col] = pd.to_numeric(onset[col], errors="coerce")

for col in mmse_numeric_cols:
    mmse[col] = pd.to_numeric(mmse[col], errors="coerce")

for col in casi_numeric_cols:
    casi[col] = pd.to_numeric(casi[col], errors="coerce")

for col in blood_numeric_cols:
    blood[col] = pd.to_numeric(blood[col], errors="coerce")

In [21]:
def set_outside_range_to_nan(df, column, lower, upper):
    invalid = ~df[column].between(lower, upper) & df[column].notna()
    
    print(
        f"{column}: 發現 {invalid.sum()} 筆超出合理範圍"
    )
    
    df.loc[invalid, column] = np.nan


# MMSE：0–30
set_outside_range_to_nan(mmse, "mmse", 0, 30)

# CDR global：0–3
set_outside_range_to_nan(mmse, "cdr_global", 0, 3)

# CDR-SOB：0–18
set_outside_range_to_nan(mmse, "cdr_sob", 0, 18)

# CASI total：0–100
set_outside_range_to_nan(casi, "casi_total", 0, 100)

# CASI 各分項
casi_ranges = {
    "casi_mental_manipulation": (0, 10),
    "casi_attention": (0, 8),
    "casi_orientation": (0, 18),
    "casi_long_term_memory": (0, 10),
    "casi_short_term_memory": (0, 12),
    "casi_abstraction": (0, 12),
    "casi_drawing": (0, 10),
    "casi_verbal_fluency": (0, 10),
    "casi_language": (0, 10)
}

for col, (lower, upper) in casi_ranges.items():
    set_outside_range_to_nan(casi, col, lower, upper)

# 一般人口學合理範圍
set_outside_range_to_nan(demo, "age_first_mmse", 18, 110)
set_outside_range_to_nan(demo, "education", 0, 30)
set_outside_range_to_nan(onset, "age_onset", 0, 110)

mmse: 發現 0 筆超出合理範圍
cdr_global: 發現 0 筆超出合理範圍
cdr_sob: 發現 0 筆超出合理範圍
casi_total: 發現 0 筆超出合理範圍
casi_mental_manipulation: 發現 0 筆超出合理範圍
casi_attention: 發現 0 筆超出合理範圍
casi_orientation: 發現 0 筆超出合理範圍
casi_long_term_memory: 發現 0 筆超出合理範圍
casi_short_term_memory: 發現 0 筆超出合理範圍
casi_abstraction: 發現 0 筆超出合理範圍
casi_drawing: 發現 0 筆超出合理範圍
casi_verbal_fluency: 發現 0 筆超出合理範圍
casi_language: 發現 0 筆超出合理範圍
age_first_mmse: 發現 0 筆超出合理範圍
education: 發現 4 筆超出合理範圍
age_onset: 發現 0 筆超出合理範圍


#### 有四筆education資料 = -1

### 極端血液數值設定為缺失值

In [22]:
blood_ranges = {
    "hdl": (1, 250),
    "vldl": (1, 200),
    "ldl": (1, 500),
    "total_cholesterol": (20, 700),
    "triglyceride": (1, 2000),
    "fasting_glucose": (1, 1000),
    "hba1c": (2, 25)
}

for col, (lower, upper) in blood_ranges.items():
    set_outside_range_to_nan(blood, col, lower, upper)

hdl: 發現 0 筆超出合理範圍
vldl: 發現 0 筆超出合理範圍
ldl: 發現 0 筆超出合理範圍
total_cholesterol: 發現 0 筆超出合理範圍
triglyceride: 發現 0 筆超出合理範圍
fasting_glucose: 發現 0 筆超出合理範圍
hba1c: 發現 0 筆超出合理範圍


#### fasting_glucose有兩筆低於20 目前先不設定NaN

### 處理同一患者同一天重複 MMSE

In [23]:
duplicate_mmse = mmse[
    mmse.duplicated(
        subset=["patient_id", "visit_date"],
        keep=False
    )
].sort_values(["patient_id", "visit_date"])

print(duplicate_mmse)

      patient_id visit_date  mmse  cdr_global  cdr_memory  cdr_orientation  \
2018         201 2022-09-06  26.0         0.5         0.5              0.0   
2019         201 2022-09-06  26.0         0.5         0.5              0.0   
2291         241 2022-09-05  29.0         0.5         0.5              0.0   
2292         241 2022-09-05  29.0         0.5         0.5              0.0   
2749         318 2022-09-07   8.0         1.0         2.0              2.0   
2750         318 2022-09-07   8.0         1.0         2.0              2.0   
2786         323 2022-09-05   0.0         2.0         3.0              3.0   
2787         323 2022-09-05   0.0         2.0         3.0              3.0   
2982         358 2022-09-05   5.0         1.0         2.0              2.0   
2983         358 2022-09-05   5.0         1.0         2.0              2.0   
3318         415 2022-09-07  20.0         0.5         1.0              0.5   
3319         415 2022-09-07  20.0         0.5         1.0       

### 對mmse long表格同一病人同一日期的資料取平均處理 (可以思考其他方法)

In [10]:
mmse = (
    mmse
    .dropna(subset=["patient_id", "visit_date", "mmse"])
    .groupby(
        ["patient_id", "visit_date"],
        as_index=False
    )[mmse_numeric_cols]
    .mean()
)

In [24]:
print("mmse資料表的病人數量:",len(mmse["patient_id"].unique()))

mmse資料表的病人數量: 1297


### casi和blood資料表沒有同病人同日期的重複資料

In [ ]:
duplicate_casi = casi[
    casi.duplicated(
        subset=["patient_id", "casi_date"],
        keep=False
    )                                                                                                                                                                              
].sort_values(["patient_id", "casi_date"])

print(duplicate_casi)

Empty DataFrame
Columns: [patient_id, casi_date, casi_mental_manipulation, casi_attention, casi_orientation, casi_long_term_memory, casi_short_term_memory, casi_abstraction, casi_drawing, casi_verbal_fluency, casi_language, casi_total]
Index: []


In [25]:
duplicate_blood = blood[
    blood.duplicated(
        subset=["patient_id", "blood_date"],
        keep=False
    )
].sort_values(["patient_id", "blood_date"])

print(duplicate_blood)

Empty DataFrame
Columns: [patient_id, blood_date, hdl, vldl, ldl, total_cholesterol, triglyceride, fasting_glucose, hba1c]
Index: []


### 建立人口學特徵

In [26]:
# ============================================================
# APOE preprocessing
# 適用後續 LSTM / NARX / TCN
# ============================================================

# ------------------------------------------------------------
# 1. 清理 APOE genotype 字串
# ------------------------------------------------------------

def clean_apoe(value):
    if pd.isna(value):
        return np.nan

    value = str(value).upper()
    value = value.replace(" ", "")
    value = value.replace("Ε", "E")

    return value


# ------------------------------------------------------------
# 2. 計算 APOE E4 allele 數量
#
# 0 = 無 E4
# 1 = 一個 E4
# 2 = 兩個 E4
# NaN = APOE unknown
# ------------------------------------------------------------

def apoe4_count(value):
    if pd.isna(value):
        return np.nan

    value = clean_apoe(value)

    return value.count("E4")


# ============================================================
# 建立原始衍生欄位
# ============================================================

demo["apoe_clean"] = (
    demo["apoe"]
    .apply(clean_apoe)
)

demo["apoe4_count"] = (
    demo["apoe_clean"]
    .apply(apoe4_count)
)


# ------------------------------------------------------------
# 3. 建立 APOE missing indicator
#
# 0 = APOE 已知
# 1 = APOE 永久缺失 / unknown
#
# 必須在 fillna 之前建立
# ------------------------------------------------------------

demo["apoe_missing"] = (
    demo["apoe4_count"]
    .isna()
    .astype(np.float32)
)


# ------------------------------------------------------------
# 4. 建立 carrier 欄位
#
# 可保留做描述統計，
# 但後續模型建議不要與 apoe4_count 同時使用
# ------------------------------------------------------------

demo["apoe4_carrier"] = np.where(
    demo["apoe4_count"].isna(),
    np.nan,
    (demo["apoe4_count"] >= 1).astype(int)
)


# ------------------------------------------------------------
# 5. 處理 apoe4_count 缺失
#
# -1 = APOE unknown
#
# 真實取值：
#  0 = 0 copies
#  1 = 1 copy
#  2 = 2 copies
# ------------------------------------------------------------

demo["apoe4_count"] = (
    demo["apoe4_count"]
    .fillna(-1)
    .astype(np.float32)
)


# ============================================================
# 檢查結果
# ============================================================

print("APOE genotype distribution:")
print(
    demo["apoe_clean"]
    .value_counts(dropna=False)
)

print("\nAPOE E4 count distribution:")
print(
    demo["apoe4_count"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nAPOE missing indicator:")
print(
    demo["apoe_missing"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nCross-check:")
print(
    pd.crosstab(
        demo["apoe4_count"],
        demo["apoe_missing"],
        margins=True
    )
)

APOE genotype distribution:
apoe_clean
E3/E3    671
E3/E4    262
NaN      194
E2/E3    114
E4/E4     39
E2/E4     17
E2/E2      4
Name: count, dtype: int64

APOE E4 count distribution:
apoe4_count
-1.0    194
 0.0    789
 1.0    279
 2.0     39
Name: count, dtype: int64

APOE missing indicator:
apoe_missing
0.0    1107
1.0     194
Name: count, dtype: int64

Cross-check:
apoe_missing   0.0  1.0   All
apoe4_count                  
-1.0             0  194   194
 0.0           789    0   789
 1.0           279    0   279
 2.0            39    0    39
 All          1107  194  1301


### 病程長度

In [27]:
# 將 onset 資料表中的發病年齡併入 demo 資料表
# 使用 patient_id 對應病人；left join 會保留 demo 中的所有病人
# 若病人在 onset 沒有發病年齡，新增的 age_onset 欄位會是 NaN
# validate="one_to_one" 用來確認兩個資料表中的 patient_id 都是唯一的，避免合併後資料筆數異常增加
demo = demo.merge(
    onset[["patient_id", "age_onset"]],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

# # 計算 baseline 時的疾病持續時間（年）
# # disease_duration_at_baseline = 第一次 MMSE 年齡 - 發病年齡
# # 例如：第一次 MMSE 時 70 歲、發病時 65 歲，病程長度為 5 年
# # 若 age_first_mmse 或 age_onset 缺失，計算結果也會是 NaN
# demo["disease_duration_at_baseline"] = (
#     demo["age_first_mmse"] - demo["age_onset"]
# )

# # 病程長度不應為負值；負值代表年齡資料可能有誤或日期/欄位定義不一致
# # 將不合理的負值設為 NaN，避免錯誤資料進入後續分析或模型
# demo.loc[
#     demo["disease_duration_at_baseline"] < 0,
#     "disease_duration_at_baseline"
# ] = np.nan

In [28]:
print("demo資料的病人數量",len(demo["patient_id"].unique()))

demo資料的病人數量 1301


### 建立MMSE預測用的標籤

- 一年後的MMSE

In [22]:
# 對病人的每一次就診 i：把這次就診日期當成 index_date
# 找出之後的所有回診 future
# 計算每次未來回診距離這次就診幾天 followup_days

# 從未來回診中，挑出「大約一年後」的候選資料：條件是 275 ~ 455 天

# 如果沒有落在這個區間的回診，就跳過這次就診。
# 如果有多筆候選，就選最接近 365 天的那一筆。
# 把這次就診的資料複製成一列，並加上：target_date：一年後目標回診日期
# target_mmse_1y：一年後的 MMSE
# target_followup_days：實際相隔天數

mmse = (
    mmse
    .sort_values(["patient_id", "visit_date"])
    .reset_index(drop=True)
)

samples = []

for patient_id, g in mmse.groupby("patient_id"):

    g = g.sort_values("visit_date").reset_index(drop=True)

    for i in range(len(g)):

        index_date = g.loc[i, "visit_date"]

        future = g.iloc[i + 1:].copy()

        future["followup_days"] = (
            future["visit_date"] - index_date
        ).dt.days

        # 約一年後：275–455 天
        candidate = future[
            future["followup_days"].between(275, 455)
        ].copy()

        if len(candidate) == 0:
            continue

        # 找最接近 365 天
        candidate["distance_to_1y"] = (
            candidate["followup_days"] - 365
        ).abs()

        target = candidate.loc[
            candidate["distance_to_1y"].idxmin()
        ]

        row = g.loc[i].copy()

        row["target_date"] = target["visit_date"]
        row["target_mmse_1y"] = target["mmse"]
        row["target_followup_days"] = target["followup_days"]

        samples.append(row)

mmse_model = pd.DataFrame(samples)
print("剩下病人數量:", len(mmse_model["patient_id"].unique()))

剩下病人數量: 723


In [17]:
mmse_model["previous_mmse"] = (
    mmse_model.groupby("patient_id")["mmse"].shift(1)
)

mmse_model["previous_visit_date"] = (
    mmse_model.groupby("patient_id")["visit_date"].shift(1)
)

mmse_model["mmse_change_prev"] = (
    mmse_model["mmse"] - mmse_model["previous_mmse"]
)

mmse_model["days_since_prev_mmse"] = (
    mmse_model["visit_date"] - mmse_model["previous_visit_date"]
).dt.days

- 用過去所有MMSE量測預測最新一次的mmse

In [23]:
# 對每位病人的每一次就診 i：
# 把目前這次 visit 當作歷史序列的一部分

# 找下一次回診 i+1

# 下一次回診的 MMSE 當作 target

# 計算目前 visit 到下一次 visit 的間隔天數

# 病人的最後一次 visit 因為沒有下一次 MMSE，
# 所以不能當 training sample

mmse = (
    mmse
    .sort_values(["patient_id", "visit_date"])
    .reset_index(drop=True)
)

samples = []

for patient_id, g in mmse.groupby("patient_id"):

    g = (
        g.sort_values("visit_date")
        .reset_index(drop=True)
    )

    # 最後一筆沒有 future target
    for i in range(len(g) - 1):

        index_date = g.loc[i, "visit_date"]

        # 下一次 MMSE
        target = g.loc[i + 1]

        followup_days = (
            target["visit_date"] - index_date
        ).days

        # 複製目前這次 visit 的資料
        row = g.loc[i].copy()

        # 加入預測目標
        row["target_date"] = target["visit_date"]
        row["target_mmse"] = target["mmse"]
        row["target_followup_days"] = followup_days

        samples.append(row)

mmse_model = pd.DataFrame(samples)
print("剩下病人數量:", len(mmse_model["patient_id"].unique()))

剩下病人數量: 915


In [24]:
mmse_model["previous_mmse"] = (
    mmse_model
    .groupby("patient_id")["mmse"]
    .shift(1)
)

mmse_model["previous_visit_date"] = (
    mmse_model
    .groupby("patient_id")["visit_date"]
    .shift(1)
)

mmse_model["mmse_change_prev"] = (
    mmse_model["mmse"]
    - mmse_model["previous_mmse"]
)

mmse_model["days_since_prev_mmse"] = (
    mmse_model["visit_date"]
    - mmse_model["previous_visit_date"]
).dt.days

print("剩下病人數量:", len(mmse_model["patient_id"].unique()))

剩下病人數量: 915


In [25]:
print("原始 MMSE 病人數：", mmse["patient_id"].nunique())
print("建立預測特徵後病人數：", mmse_model["patient_id"].nunique())

print(
    "被排除病人數：",
    mmse["patient_id"].nunique()
    - mmse_model["patient_id"].nunique()
)

原始 MMSE 病人數： 1297
建立預測特徵後病人數： 915
被排除病人數： 382


- 用前面所有的mmse預測最後一次mmse

In [30]:
mmse_model = mmse

### 合併人口學資料，將demo和mmse_model表合併成model_df

In [ ]:
# "disease_duration_at_baseline" 先不放，因為該欄位目前有問題
demo_features = [
    "patient_id",
    "diagnosis",
    "bio_category",
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia",
    "apoe_clean",
    "apoe4_count",
    "apoe4_carrier",
    "age_onset",
    
]

model_df = mmse_model.merge(
    demo[demo_features],
    on="patient_id",
    how="left",
    validate="many_to_one"
)

print("剩下病人數量:", len(model_df["patient_id"].unique()))

剩下病人數量: 1297


### 合併 CASI

In [33]:
# ============================================================
# CASI
# 優先使用：
# 1. MMSE 同一天的 CASI
# 2. 若沒有，使用 MMSE 前 180 天內最近一次 CASI
# 3. 不使用 MMSE 之後的 CASI，避免未來資料洩漏
# ============================================================

model_df = model_df.dropna(
    subset=["patient_id", "visit_date"]
).copy()

casi_for_merge = casi.dropna(
    subset=["patient_id", "casi_date"]
).copy()


# 統一 patient_id 型別
model_df["patient_id"] = (
    model_df["patient_id"].astype("int64")
)

casi_for_merge["patient_id"] = (
    casi_for_merge["patient_id"].astype("int64")
)


# merge_asof 要依 asof key 排序
model_df = model_df.sort_values(
    ["visit_date", "patient_id"]
).reset_index(drop=True)

casi_for_merge = casi_for_merge.sort_values(
    ["casi_date", "patient_id"]
).reset_index(drop=True)


# 合併 CASI
model_df = pd.merge_asof(
    model_df,
    casi_for_merge,
    left_on="visit_date",
    right_on="casi_date",
    by="patient_id",
    direction="backward",
    tolerance=pd.Timedelta(days=180)
)


# MMSE 與 CASI 實際相隔幾天
model_df["days_since_casi"] = (
    model_df["visit_date"]
    - model_df["casi_date"]
).dt.days


# 是否有近期 CASI
model_df["has_recent_casi"] = (
    model_df["casi_date"]
    .notna()
    .astype(int)
)

print("剩下病人數量:", len(model_df["patient_id"].unique()))

剩下病人數量: 1297


### 合併血液資料

In [34]:
import pandas as pd
import numpy as np


# ============================================================
# 1. 基本資料格式處理
# ============================================================

model_df = model_df.copy()
blood_temp = blood.copy()

model_df["patient_id"] = pd.to_numeric(
    model_df["patient_id"],
    errors="coerce"
)

blood_temp["patient_id"] = pd.to_numeric(
    blood_temp["patient_id"],
    errors="coerce"
)

model_df["visit_date"] = pd.to_datetime(
    model_df["visit_date"],
    errors="coerce"
)

blood_temp["blood_date"] = pd.to_datetime(
    blood_temp["blood_date"],
    errors="coerce"
)


# 移除沒有 ID / 日期，無法配對的資料
model_df = model_df.dropna(
    subset=["patient_id", "visit_date"]
).copy()

blood_temp = blood_temp.dropna(
    subset=["patient_id", "blood_date"]
).copy()


model_df["patient_id"] = model_df["patient_id"].astype("int64")
blood_temp["patient_id"] = blood_temp["patient_id"].astype("int64")

# ============================================================
# 2. 找出 blood feature
# ============================================================

exclude_cols = [
    "patient_id",
    "blood_date"
]

blood_features = [
    col for col in blood_temp.columns
    if col not in exclude_cols
]

print("Blood features:")
print(blood_features)

print("\n總共:", len(blood_features), "個欄位")

# ============================================================
# 3. 每個 blood feature
#    各自找同病人、距離 MMSE 最近的非缺失值
# ============================================================

result_df = model_df.copy()

# 保留原始 row id，最後恢復原本順序
result_df["_row_id"] = np.arange(len(result_df))


for feature in blood_features:

    print(f"Processing: {feature}")

    # --------------------------------------------------------
    # 只留下「這個血液指標有值」的紀錄
    # --------------------------------------------------------

    feature_blood = blood_temp[
        ["patient_id", "blood_date", feature]
    ].dropna(
        subset=[feature]
    ).copy()


    # 如果這個 feature 完全沒有資料就跳過
    if feature_blood.empty:
        print(f"  {feature}: 無有效資料，跳過")
        result_df[feature] = np.nan
        result_df[f"{feature}_blood_date"] = pd.NaT
        result_df[f"{feature}_gap_days"] = np.nan
        continue


    # --------------------------------------------------------
    # merge_asof 需要排序
    # --------------------------------------------------------

    left = result_df[
        ["_row_id", "patient_id", "visit_date"]
    ].sort_values(
        ["visit_date", "patient_id"]
    ).copy()


    right = feature_blood.sort_values(
        ["blood_date", "patient_id"]
    ).copy()


    # --------------------------------------------------------
    # 找「最近日期」
    #
    # direction="nearest"
    # → MMSE 日期之前或之後都可以
    # --------------------------------------------------------

    matched = pd.merge_asof(
        left,
        right,
        left_on="visit_date",
        right_on="blood_date",
        by="patient_id",
        direction="nearest"
    )


    # --------------------------------------------------------
    # 重新命名，避免不同 feature 的日期互相覆蓋
    # --------------------------------------------------------

    matched = matched.rename(
        columns={
            feature: f"{feature}_matched",
            "blood_date": f"{feature}_blood_date"
        }
    )


    # --------------------------------------------------------
    # MMSE 與 blood 的日期距離
    # --------------------------------------------------------

    matched[f"{feature}_gap_days"] = (
        matched[f"{feature}_blood_date"]
        - matched["visit_date"]
    ).dt.days


    # --------------------------------------------------------
    # 合併回 result_df
    # --------------------------------------------------------

    result_df = result_df.merge(
        matched[
            [
                "_row_id",
                f"{feature}_matched",
                f"{feature}_blood_date",
                f"{feature}_gap_days"
            ]
        ],
        on="_row_id",
        how="left"
    )


    # 真正要用的 blood feature
    result_df[feature] = result_df[
        f"{feature}_matched"
    ]

    result_df = result_df.drop(
        columns=[f"{feature}_matched"]
    )

# ============================================================
# 4. 恢復原本順序
# ============================================================

result_df = (
    result_df
    .sort_values("_row_id")
    .drop(columns="_row_id")
    .reset_index(drop=True)
)

model_df = result_df

print("剩下病人數量:", len(model_df["patient_id"].unique()))

Blood features:
['hdl', 'vldl', 'ldl', 'total_cholesterol', 'triglyceride', 'fasting_glucose', 'hba1c']

總共: 7 個欄位
Processing: hdl
Processing: vldl
Processing: ldl
Processing: total_cholesterol
Processing: triglyceride
Processing: fasting_glucose
Processing: hba1c
剩下病人數量: 1297


In [35]:
summary = []

for feature in blood_features:

    n = model_df[feature].notna().sum()
    pct = model_df[feature].notna().mean() * 100

    median_gap = model_df.loc[
        model_df[feature].notna(),
        f"{feature}_gap_days"
    ].abs().median()

    summary.append({
        "feature": feature,
        "available_n": n,
        "available_pct": pct,
        "median_gap_days": median_gap
    })


blood_merge_summary = pd.DataFrame(summary)

blood_merge_summary = blood_merge_summary.sort_values(
    "available_pct",
    ascending=False
)

blood_merge_summary

,feature,available_n,available_pct,median_gap_days
3,total_cholesterol,4639,83.585586,22.0
4,triglyceride,4628,83.387387,22.0
0,hdl,4628,83.387387,22.0
6,hba1c,4628,83.387387,22.0
2,ldl,4601,82.900901,24.0
1,vldl,4397,79.225225,28.0
5,fasting_glucose,4396,79.207207,28.0


### model_df的缺失欄位狀況

In [36]:
missing_summary = pd.DataFrame({
    "missing_count": model_df.isna().sum(),
    "missing_rate_pct": model_df.isna().mean() * 100
})

missing_summary = (
    missing_summary
    .query("missing_count > 0")
    .sort_values("missing_rate_pct", ascending=False)
)

missing_summary["missing_rate_pct"] = (
    missing_summary["missing_rate_pct"].round(2)
)

missing_summary

,missing_count,missing_rate_pct
fasting_glucose_blood_date,1154,20.79
fasting_glucose_gap_days,1154,20.79
fasting_glucose,1154,20.79
vldl_gap_days,1153,20.77
vldl,1153,20.77
vldl_blood_date,1153,20.77
casi_date,1099,19.80
casi_attention,1099,19.80
casi_mental_manipulation,1099,19.80
casi_abstraction,1099,19.80


In [37]:
casi_cols = [
    "casi_verbal_fluency",
    "casi_drawing",
    "casi_abstraction",
    "casi_short_term_memory",
    "casi_long_term_memory",
    "casi_orientation",
    "casi_attention",
    "casi_mental_manipulation",
    "casi_language",
    "casi_total"
]

model_df[casi_cols].isna().all(axis=1).value_counts()

missing_casi = model_df[
    model_df[casi_cols].isna().all(axis=1)
]

print("缺 CASI 的 observations:", len(missing_casi))
print(
    "涉及病人數:",
    missing_casi["patient_id"].nunique()
)

# 每位病人的 CASI 缺失比例
casi_missing_by_patient = (
    model_df
    .assign(
        casi_missing=model_df["casi_total"].isna()
    )
    .groupby("patient_id")["casi_missing"]
    .agg(["sum", "count", "mean"])
    .sort_values("mean", ascending=False)
)

casi_missing_by_patient

缺 CASI 的 observations: 1099
涉及病人數: 674


,sum,count,mean
patient_id,,,
6,1,1,1.0
1301,1,1,1.0
1285,1,1,1.0
501,1,1,1.0
502,1,1,1.0
...,...,...,...
1280,0,1,0.0
1281,0,1,0.0
1282,0,1,0.0


In [38]:
model_df

,patient_id,visit_date,mmse,cdr_global,cdr_memory,cdr_orientation,cdr_judgment,cdr_community,cdr_home_hobbies,cdr_personal_care,...,total_cholesterol,triglyceride_blood_date,triglyceride_gap_days,triglyceride,fasting_glucose_blood_date,fasting_glucose_gap_days,fasting_glucose,hba1c_blood_date,hba1c_gap_days,hba1c
0,9,2003-12-24,18.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,...,171.0,2007-11-12,1419.0,299.0,2014-05-03,3783.0,111.0,2012-01-09,2938.0,5.5
1,11,2004-02-03,19.0,0.5,1.0,0.5,0.5,0.5,0.5,1.0,...,228.0,2007-02-22,1115.0,108.0,2016-04-07,4447.0,123.0,2008-05-10,1558.0,5.9
2,2,2004-07-20,23.0,0.5,0.5,1.0,0.0,0.5,0.0,0.0,...,223.0,2006-07-27,737.0,153.0,NaT,NaN,NaN,2013-08-03,3301.0,7.6
3,9,2004-12-20,10.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,...,171.0,2007-11-12,1057.0,299.0,2014-05-03,3421.0,111.0,2012-01-09,2576.0,5.5
4,9,2005-01-17,16.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,...,171.0,2007-11-12,1029.0,299.0,2014-05-03,3393.0,111.0,2012-01-09,2548.0,5.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5545,911,2025-03-28,17.0,0.5,1.0,0.5,0.5,0.5,0.5,0.0,...,193.0,2025-03-28,0.0,150.0,2025-03-28,0.0,143.0,2025-03-28,0.0,6.6
5546,1124,2025-03-28,24.0,0.5,0.5,0.0,0.5,0.0,0.0,0.0,...,193.0,2025-03-28,0.0,244.0,2025-03-28,0.0,157.0,2025-03-28,0.0,7.1
5547,1128,2025-03-28,30.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,...,160.0,2024-04-16,-346.0,93.0,2024-04-16,-346.0,126.0,2024-04-16,-346.0,6.0
5548,227,2025-03-31,0.0,2.0,2.0,2.0,2.0,2.0,2.0,3.0,...,151.0,2023-03-17,-745.0,85.0,2023-03-17,-745.0,120.0,2023-03-17,-745.0,5.8


In [24]:
model_df.to_csv("../data/processed/model_df.csv", index=False)

In [33]:
model_df.to_csv("../data/processed/model_df_next_mmse.csv", index=False)

In [39]:
model_df.to_csv("../data/processed/model_df_next_mmse3.csv", index=False)

### 建立模型輸入與目標

In [64]:
target = "target_mmse_1y"

exclude_cols = [
    "patient_id",
    "visit_date",

    "target_date",
    "target_mmse_1y",
    "target_followup_days",

    "previous_visit_date",
    "casi_date",
    "blood_date",

    # 暫時排除，確認是否有 future information 後再決定
    "diagnosis",
    "bio_category",
]

X = model_df.drop(columns=exclude_cols)
y = model_df[target]

In [66]:
X.columns

Index(['mmse', 'cdr_global', 'cdr_memory', 'cdr_orientation', 'cdr_judgment',
       'cdr_community', 'cdr_home_hobbies', 'cdr_personal_care', 'cdr_sob',
       'previous_mmse', 'mmse_change_prev', 'days_since_prev_mmse', 'gender',
       'age_first_mmse', 'education', 'hypertension', 'diabetes',
       'hyperlipidemia', 'apoe_clean', 'apoe4_count', 'apoe4_carrier',
       'age_onset', 'disease_duration_at_baseline', 'casi_mental_manipulation',
       'casi_attention', 'casi_orientation', 'casi_long_term_memory',
       'casi_short_term_memory', 'casi_abstraction', 'casi_drawing',
       'casi_verbal_fluency', 'casi_language', 'casi_total', 'days_since_casi',
       'has_recent_casi', 'hdl', 'vldl', 'ldl', 'total_cholesterol',
       'triglyceride', 'fasting_glucose', 'hba1c', 'days_since_blood',
       'has_recent_blood'],
      dtype='object')

In [67]:
combined_Xy = pd.concat([X, y], axis=1)

In [68]:
combined_Xy.head()

,mmse,cdr_global,cdr_memory,cdr_orientation,cdr_judgment,cdr_community,cdr_home_hobbies,cdr_personal_care,cdr_sob,previous_mmse,...,hdl,vldl,ldl,total_cholesterol,triglyceride,fasting_glucose,hba1c,days_since_blood,has_recent_blood,target_mmse_1y
0,18.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,4.5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,10.0
1,10.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,4.5,18.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,362.0,1,10.0
2,16.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,4.5,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,12.0
3,10.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,4.5,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,13.0
4,12.0,1.0,1.0,1.0,1.0,1.0,0.5,0.0,4.5,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,14.0
